# 01 — Génération des métriques · Nowledgeable

Ce notebook exécute **uniquement** les indicateurs déclarés dans `metric_registry.py`
pour le corpus **Nowledgeable**. Les sorties sont isolées dans `csv/Nowledgeable/` et les
journaux dans `csv/Nowledgeable/logs/`.

Tous les scripts sont lancés dans des sous-processus indépendants afin d'éviter les
effets de bord entre imports, loggers et arguments de ligne de commande.


## 1. Configuration


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from metric_registry import get_dataset
from pipeline_utils import detect_project_dir, inspect_input, run_metric_jobs, metric_inventory

PROJECT_DIR = detect_project_dir()
DATASET = "Nowledgeable"
SPEC = get_dataset(DATASET)

# Laisser à None pour utiliser le chemin défini dans metric_registry.py.
# Exemple Mirabelle : PROJECT_DIR / "data" / "autres_traces.csv"
# Exemple ProgSnap2 : PROJECT_DIR / "data"  (dossier contenant MainTable.csv)
INPUT_OVERRIDE = None

OVERWRITE_OUTPUTS = True

print("Projet       :", PROJECT_DIR)
print("Corpus       :", DATASET)
print("Scripts      :", SPEC.scripts_path(PROJECT_DIR))
print("Sorties CSV  :", SPEC.csv_dir(PROJECT_DIR))
print("Entrée défaut:", SPEC.input(PROJECT_DIR))


Projet       : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine nettoyée\Chaine_nettoyee
Corpus       : Nowledgeable
Scripts      : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine nettoyée\Chaine_nettoyee\scripts_Nowledgeable
Sorties CSV  : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine nettoyée\Chaine_nettoyee\csv\Nowledgeable
Entrée défaut: C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine nettoyée\Chaine_nettoyee\data\session_13568_answers_corrige.csv


## 2. Contrôle de l'entrée


In [2]:
input_df, input_summary = inspect_input(PROJECT_DIR, SPEC, INPUT_OVERRIDE)
display(input_summary)
print("Colonnes disponibles :")
print(", ".join(input_df.columns.astype(str)))


,corpus,fichier,lignes,colonnes,sujets_uniques,colonne_identifiant
0,Nowledgeable,session_13568_answers_corrige.csv,1422,14,30,studentId


Colonnes disponibles :
answerUuid, studentId, exerciceId, exerciceType, exerciceTitle, answeredAt, answerContent, answerScore, answerIsRight, assessmentUuid, assessmentScore, assessmentIsManual, assessmentStatus, recordedFeedback


## 3. Indicateurs déclarés


In [3]:
jobs_df = pd.DataFrame([
    {
        "script": item.script,
        "sortie": item.output,
        "métrique": item.metric,
        "description": item.description,
        "colonnes requises": ", ".join(item.required_columns),
        "arguments": " ".join(item.extra_args),
    }
    for item in SPEC.jobs
])
display(jobs_df)


,script,sortie,métrique,description,colonnes requises,arguments
0,session_count_Nowledgeable.py,session_count.csv,SessionCount,Nombre de sessions d'activité séparées par plu...,"studentId, answeredAt",5.0
1,test_pass_rate_Nowledgeable.py,test_pass_rate.csv,TestPassRate,Proportion des cas de test enseignants réussis.,"studentId, recordedFeedback",
2,total_test_count_Nowledgeable.py,total_test_count.csv,TotalTestCount,Nombre total d'exécutions de cas de test détai...,"studentId, recordedFeedback",
3,exercise_coverage_Nowledgeable.py,exercise_coverage.csv,ExerciseCoverage,Nombre d'exercices considérés comme traités.,"studentId, exerciceId, answerScore, answerContent",
4,function_coverage_Nowledgeable.py,function_coverage.csv,FunctionCoverage,Nombre de fonctions C/C++ considérées comme tr...,"studentId, exerciceId, answerScore, answerContent",
5,score_progression_Nowledgeable.py,score_progression.csv,ScoreProgression,Progression moyenne entre le premier et le der...,"studentId, exerciceId, answeredAt, answerScore",
6,max_unchanged_code_attempts_Nowledgeable.py,max_unchanged_code_attempts.csv,MaxUnchangedCodeAttempts,Plus longue série de soumissions consécutives ...,"studentId, exerciceId, answeredAt, answerContent",
7,eq_Nowledgeable.py,error_quotient.csv,ErrorQuotient,Error Quotient calculé sur les diagnostics de ...,"studentId, exerciceId, answeredAt, recordedFee...",
8,red_Nowledgeable.py,red.csv,RED,Repeated Error Density sur les diagnostics de ...,"studentId, exerciceId, answeredAt, recordedFee...",
9,attempts_to_first_success_Nowledgeable.py,attempts_to_first_success.csv,AttemptsToFirstSuccess,Nombre moyen de tentatives jusqu'au premier ex...,"studentId, exerciceId, answeredAt, answerIsRight",


## 4. Exécution


In [4]:
report_df = run_metric_jobs(
    PROJECT_DIR,
    DATASET,
    overwrite=OVERWRITE_OUTPUTS,
    input_override=INPUT_OVERRIDE,
)
display(report_df[[
    "script", "metric", "status", "rows", "columns", "message", "log_path"
]])

errors = report_df[~report_df["status"].isin(["ok", "skipped_existing"])]
if errors.empty:
    print("Tous les indicateurs ont été générés correctement.")
else:
    print(f"{len(errors)} indicateur(s) à vérifier. Consultez les fichiers log_path.")


,script,metric,status,rows,columns,message,log_path
0,session_count_Nowledgeable.py,SessionCount,ok,30,"SubjectID, SessionCount",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
1,test_pass_rate_Nowledgeable.py,TestPassRate,ok,22,"SubjectID, TestPassRate",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
2,total_test_count_Nowledgeable.py,TotalTestCount,ok,30,"SubjectID, TotalTestCount",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
3,exercise_coverage_Nowledgeable.py,ExerciseCoverage,ok,30,"SubjectID, ExerciseCoverage",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
4,function_coverage_Nowledgeable.py,FunctionCoverage,ok,30,"SubjectID, FunctionCoverage",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
5,score_progression_Nowledgeable.py,ScoreProgression,ok,30,"SubjectID, ScoreProgression",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
6,max_unchanged_code_attempts_Nowledgeable.py,MaxUnchangedCodeAttempts,ok,30,"SubjectID, MaxUnchangedCodeAttempts",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
7,eq_Nowledgeable.py,ErrorQuotient,ok,28,"SubjectID, ErrorQuotient",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
8,red_Nowledgeable.py,RED,ok,28,"SubjectID, RED",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...
9,attempts_to_first_success_Nowledgeable.py,AttemptsToFirstSuccess,ok,23,"SubjectID, AttemptsToFirstSuccess",CSV produit et validé,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...


Tous les indicateurs ont été générés correctement.


## 5. Inventaire des sorties


In [5]:
inventory_df = metric_inventory(SPEC.csv_dir(PROJECT_DIR))
display(inventory_df)


,fichier,statut,lignes,colonnes,variables
0,attempts_to_first_success.csv,ok,23,2,AttemptsToFirstSuccess
1,code_change_magnitude.csv,ok,29,2,CodeChangeMagnitude
2,error_quotient.csv,ok,28,2,ErrorQuotient
3,exercise_coverage.csv,ok,30,2,ExerciseCoverage
4,first_attempt_success_rate.csv,ok,30,2,FirstAttemptSuccessRate
5,function_coverage.csv,ok,30,2,FunctionCoverage
6,max_unchanged_code_attempts.csv,ok,30,2,MaxUnchangedCodeAttempts
7,productive_transition_rate.csv,ok,29,2,ProductiveTransitionRate
8,red.csv,ok,28,2,RED
9,score_progression.csv,ok,30,2,ScoreProgression
